# MobileNetV2 Transfer Learning

## MobileNetV2 Transfer Learning Introduction

The baseline CNN is trained from scratch on Mel Spectrogram tensors. Transfer learning provides an alternative approach by reusing visual features learned from a large image dataset.

MobileNetV2 is a compact convolutional architecture designed for efficient image classification. Although Mel Spectrograms are not natural photographs, they contain local shapes, edges, and textures that can benefit from pretrained convolutional filters.

This notebook loads the Mel Spectrogram datasets generated in Notebook 03. It uses the existing combined GTZAN + FMA Medium train / validation / test strategy and does not recompute audio features.

## Hypothesis

**H6 — Transfer Learning Advantage:** MobileNetV2 transfer learning will achieve a higher validation macro F1 score than the baseline CNN, or comparable performance within 0.05 macro F1 using fewer training epochs.

Macro F1 is the primary comparison metric because the combined dataset is imbalanced.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

from src.data.feature_extraction import load_mel_dataset
from src.models.mobilenet_model import create_mobilenet_model
from src.training.evaluate import evaluate_torch_model
from src.training.train import (
    save_model,
    save_torch_model,
    train_torch_model,
)
from src.utils.config import (
    BEST_MOBILENET_MODEL_PATH,
    CNN_METRICS_SUMMARY_PATH,
    CNN_TRAINING_HISTORY_PATH,
    MEL_TEST_LABELS_PATH,
    MEL_TEST_PATH,
    MEL_TRAIN_LABELS_PATH,
    MEL_TRAIN_PATH,
    MEL_VALIDATION_LABELS_PATH,
    MEL_VALIDATION_PATH,
    MOBILENET_BATCH_SIZE,
    MOBILENET_EPOCHS,
    MOBILENET_LABEL_ENCODER_PATH,
    MOBILENET_LEARNING_RATE,
    MOBILENET_METRICS_SUMMARY_PATH,
    MOBILENET_MODEL_DIR,
    MOBILENET_OUTPUT_DIR,
    MOBILENET_TEST_CONFUSION_MATRIX_PATH,
    MOBILENET_TEST_REPORT_PATH,
    MOBILENET_TRAINING_HISTORY_PATH,
)

## Load Mel Spectrogram Datasets

The saved train, validation, and test Mel tensors are loaded directly. The label encoder is fitted on the training labels and reused for the remaining splits.

In [ ]:
X_train, labels_train = load_mel_dataset(
    MEL_TRAIN_PATH,
    MEL_TRAIN_LABELS_PATH,
)

X_validation, labels_validation = load_mel_dataset(
    MEL_VALIDATION_PATH,
    MEL_VALIDATION_LABELS_PATH,
)

X_test, labels_test = load_mel_dataset(
    MEL_TEST_PATH,
    MEL_TEST_LABELS_PATH,
)

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(labels_train)
y_validation = label_encoder.transform(labels_validation)
y_test = label_encoder.transform(labels_test)

class_names = list(label_encoder.classes_)

print("Train tensors:", X_train.shape)
print("Validation tensors:", X_validation.shape)
print("Test tensors:", X_test.shape)
print("Classes:", class_names)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Prepare MobileNetV2 Inputs

The cached Mel tensors use the PyTorch shape `(N, 1, 128, frames)`.

MobileNetV2 expects three-channel image-like input. The reusable model wrapper performs the required conversion for each batch:

- repeats the single Mel channel three times,
- resizes each spectrogram to `224 × 224`,
- applies ImageNet normalization.

This conversion happens inside the model and avoids creating a second three-channel copy of the complete dataset in memory.

## Build MobileNetV2 Model

The model factory loads ImageNet-pretrained MobileNetV2 weights and replaces the classifier for the project genre classes.

The pretrained feature extractor is frozen for this baseline experiment. Only the new classifier head is trained. The first model construction may download the pretrained weights if they are not already cached.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

mobilenet_model = create_mobilenet_model(
    input_shape=X_train.shape[1:],
    num_classes=len(class_names),
    learning_rate=MOBILENET_LEARNING_RATE,
    train_base=False,
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in mobilenet_model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in mobilenet_model.parameters()
)

print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)
mobilenet_model

## Training

Training uses Adam, cross-entropy loss, batch loading, validation-loss monitoring, and early stopping.

Only individual batches are moved to GPU. If CUDA memory is insufficient for the configured batch size of 32, `MOBILENET_BATCH_SIZE` can be reduced to 16 in `config.py`.

In [ ]:
trained_mobilenet, training_history = train_torch_model(
    model=mobilenet_model,
    X_train=X_train,
    y_train=y_train,
    X_validation=X_validation,
    y_validation=y_validation,
    batch_size=MOBILENET_BATCH_SIZE,
    epochs=MOBILENET_EPOCHS,
    learning_rate=MOBILENET_LEARNING_RATE,
    patience=5,
)

history = pd.DataFrame(training_history)
history.tail()

## Training History Visualization

The training curves show convergence speed and the relationship between training and validation performance.

In [ ]:
fig_history, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["validation_loss"], label="validation")
axes[0].set_title("MobileNetV2 Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history["epoch"], history["train_accuracy"], label="train")
axes[1].plot(history["epoch"], history["validation_accuracy"], label="validation")
axes[1].set_title("MobileNetV2 Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## Validation Evaluation

Validation macro F1 is the primary transfer-learning performance measure. Accuracy, weighted F1, and the per-class report provide additional context.

In [ ]:
validation_result = evaluate_torch_model(
    trained_mobilenet,
    X_validation,
    y_validation,
    class_names,
    model_name="mobilenet_v2",
    split="validation",
    batch_size=MOBILENET_BATCH_SIZE,
)

validation_summary = pd.DataFrame([
    validation_result["summary"]
])

validation_summary

In [ ]:
pd.DataFrame(
    validation_result["classification_report"]
).transpose()

## Test Evaluation

The test set is evaluated once after the MobileNetV2 configuration has been trained and reviewed on validation data.

In [ ]:
test_result = evaluate_torch_model(
    trained_mobilenet,
    X_test,
    y_test,
    class_names,
    model_name="mobilenet_v2",
    split="test",
    batch_size=MOBILENET_BATCH_SIZE,
)

test_summary = pd.DataFrame([
    test_result["summary"]
])

test_summary

In [ ]:
test_report = pd.DataFrame(
    test_result["classification_report"]
).transpose()

test_report

In [ ]:
fig_confusion, ax = plt.subplots(figsize=(9, 7))

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=test_result["confusion_matrix"],
    display_labels=class_names,
)

display_matrix.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False,
)

ax.set_title("Test Confusion Matrix - MobileNetV2")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Comparison with CNN Baseline

If Notebook 05 artifacts are available, the CNN and MobileNetV2 validation/test macro F1 scores are compared directly.

H6 is supported when MobileNetV2 has higher validation macro F1, or when it is within 0.05 of the CNN result while using fewer training epochs.

In [ ]:
comparison_rows = [
    validation_result["summary"],
    test_result["summary"],
]

cnn_epochs = None

if CNN_METRICS_SUMMARY_PATH.exists():
    cnn_metrics = pd.read_csv(CNN_METRICS_SUMMARY_PATH)
    comparison_rows.extend(
        cnn_metrics[
            cnn_metrics["split"].isin(["validation", "test"])
        ].to_dict("records")
    )

if CNN_TRAINING_HISTORY_PATH.exists():
    cnn_history = pd.read_csv(CNN_TRAINING_HISTORY_PATH)
    cnn_epochs = len(cnn_history)

comparison = pd.DataFrame(comparison_rows)
comparison

In [ ]:
cnn_validation = comparison[
    (comparison["model"] == "cnn_baseline")
    & (comparison["split"] == "validation")
]

mobilenet_validation_f1 = validation_result["summary"]["macro_f1"]
mobilenet_epochs = len(history)

if cnn_validation.empty:
    print("CNN baseline metrics are not available. Run Notebook 05 before assessing H6.")
else:
    cnn_validation_f1 = cnn_validation.iloc[0]["macro_f1"]
    higher_f1 = mobilenet_validation_f1 > cnn_validation_f1
    comparable_f1 = mobilenet_validation_f1 >= cnn_validation_f1 - 0.05
    fewer_epochs = (
        cnn_epochs is not None
        and mobilenet_epochs < cnn_epochs
    )
    h6_supported = higher_f1 or (comparable_f1 and fewer_epochs)

    print("CNN validation macro F1:", cnn_validation_f1)
    print("MobileNetV2 validation macro F1:", mobilenet_validation_f1)
    print("CNN training epochs:", cnn_epochs)
    print("MobileNetV2 training epochs:", mobilenet_epochs)
    print("H6 supported:", h6_supported)

## Save Model and Artifacts

The trained MobileNetV2 weights, label encoder, training history, metric summary, test classification report, and test confusion matrix are saved under the configured artifact paths.

In [ ]:
MOBILENET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MOBILENET_MODEL_DIR.mkdir(parents=True, exist_ok=True)

save_torch_model(
    trained_mobilenet,
    BEST_MOBILENET_MODEL_PATH,
)

save_model(
    label_encoder,
    MOBILENET_LABEL_ENCODER_PATH,
)

history.to_csv(
    MOBILENET_TRAINING_HISTORY_PATH,
    index=False,
)

mobilenet_metrics = pd.concat(
    [validation_summary, test_summary],
    ignore_index=True,
)

mobilenet_metrics.to_csv(
    MOBILENET_METRICS_SUMMARY_PATH,
    index=False,
)

test_report_to_save = test_report.copy()
test_report_to_save.insert(0, "label", test_report_to_save.index)
test_report_to_save.reset_index(drop=True).to_csv(
    MOBILENET_TEST_REPORT_PATH,
    index=False,
)

fig_confusion.savefig(
    MOBILENET_TEST_CONFUSION_MATRIX_PATH,
    dpi=150,
    bbox_inches="tight",
)

print("Saved MobileNetV2 model:", BEST_MOBILENET_MODEL_PATH)
print("Saved label encoder:", MOBILENET_LABEL_ENCODER_PATH)
print("Saved training history:", MOBILENET_TRAINING_HISTORY_PATH)
print("Saved metrics summary:", MOBILENET_METRICS_SUMMARY_PATH)
print("Saved test report:", MOBILENET_TEST_REPORT_PATH)
print("Saved test confusion matrix:", MOBILENET_TEST_CONFUSION_MATRIX_PATH)

## Conclusion

This notebook applies ImageNet-pretrained MobileNetV2 to the saved Mel Spectrogram datasets while preserving the existing combined GTZAN + FMA Medium split strategy.

The experiment reports validation and test accuracy, macro F1, weighted F1, classification reports, and confusion matrix. The comparison section assesses whether transfer learning supports H6 through improved validation macro F1 or comparable performance with fewer epochs.